# Week 4 EDA & Preprocessing
This notebook performs the final data cleaning and preprocessing for the Week 4 deliverable, following the course syllabus. It loads the cleaned dataset from Week 3, applies outlier treatment, categorical encoding, numerical scaling, and feature engineering, and documents all decisions. The final model-ready dataset is saved to `data/processed/dataset_clean_v2.csv`.

In [1]:
# Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder

## 1. Load Cleaned Data
Load the cleaned dataset from Week 3.

In [2]:
df = pd.read_csv('../data/processed/remissions_db_cleaned.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (340220, 15)


,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,2020,51013232,2020-02-04,1073,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
1,2020,51013233,2020-02-04,1073,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:22,2020-02-04 08:26:06,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
2,2020,51013235,2020-02-04,1028,2020-02-04 08:00:00,9631,510,2.5,2020-02-04 07:37:56,2020-02-04 08:55:49,78,ONE TIME ESP ANGELICA RIVERA GOMEZ,ROMERO JULIA,CALLE MANUEL BECERRA 14515 COL ALAMEDAS,CH-F5
3,2020,51013236,2020-02-04,1005,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:36,2020-02-04 09:47:43,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
4,2020,51013238,2020-02-04,1026,2020-02-04 08:30:00,10145,510,3.5,2020-02-04 08:09:01,2020-02-04 09:09:15,60,ONE TIME CONSTRUCENTRO CHIH,MOLINA BALDERRAMA CESAR,PERIF DE LA JUVENTUD 9926 COL RESIDENCIA,CH-L5


## 2. Outlier Detection & Treatment
For each numerical column, outliers are detected and treated according to the syllabus. The method and rationale are documented.

In [3]:
# Identify numerical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('Numerical columns:', num_cols)

Numerical columns: ['Year', 'tkt_code', 'order_code', 'truck_code', 'ship_plant_code', 'u_Volumen', 'u_Cicle']


In [4]:
# Outlier detection using IQR rule
outlier_report = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = {'outliers': outliers, 'total': df.shape[0], 'percent': outliers / df.shape[0] * 100, 'lower': lower, 'upper': upper}
outlier_report

{'Year': {'outliers': np.int64(0),
  'total': 340220,
  'percent': np.float64(0.0),
  'lower': np.float64(2016.5),
  'upper': np.float64(2028.5)},
 'tkt_code': {'outliers': np.int64(46732),
  'total': 340220,
  'percent': np.float64(13.735818000117572),
  'lower': np.float64(50529494.5),
  'upper': np.float64(52122396.5)},
 'order_code': {'outliers': np.int64(4356),
  'total': 340220,
  'percent': np.float64(1.2803480101111047),
  'lower': np.float64(840.5),
  'upper': np.float64(1508.5)},
 'truck_code': {'outliers': np.int64(0),
  'total': 340220,
  'percent': np.float64(0.0),
  'lower': np.float64(529.0),
  'upper': np.float64(15921.0)},
 'ship_plant_code': {'outliers': np.int64(46732),
  'total': 340220,
  'percent': np.float64(13.735818000117572),
  'lower': np.float64(505.0),
  'upper': np.float64(521.0)},
 'u_Volumen': {'outliers': np.int64(0),
  'total': 340220,
  'percent': np.float64(0.0),
  'lower': np.float64(-3.75),
  'upper': np.float64(10.25)},
 'u_Cicle': {'outliers': np

### Outlier Treatment Strategy
For each variable, choose a strategy based on the outlier report and domain knowledge. Document the decision.

In [5]:
# Example: Cap outliers for all numerical columns (Winsorize at 1st and 99th percentiles)
for col in num_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower, upper)
# Documented in the article draft below

## 3. Categorical Encoding
Categorical variables are encoded according to their cardinality and type. Each decision is documented.

In [6]:
# Identify categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical columns:', cat_cols)

Categorical columns: ['order_date', 'start_time', 'typed_time', 'at_plant_time', 'name', 'Nombre del proyecto', 'ship_addr_line', 'map_page']


C:\Users\gilin\AppData\Local\Temp\ipykernel_47296\2972134890.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()


In [7]:
# Example encoding: One-hot for low-cardinality, label for binary, target for high-cardinality
encoded_df = df.copy()
encoding_report = {}
for col in cat_cols:
    n_unique = df[col].nunique()
    if n_unique == 2:
        le = LabelEncoder()
        encoded_df[col + '_le'] = le.fit_transform(df[col].astype(str))
        encoding_report[col] = 'Label encoding (binary)'
    elif n_unique <= 10:
        dummies = pd.get_dummies(df[col], prefix=col)
        encoded_df = pd.concat([encoded_df, dummies], axis=1)
        encoding_report[col] = 'One-hot encoding'
    elif n_unique > 10:
        # For demonstration, use ordinal encoding (replace with target encoding for supervised tasks)
        oe = OrdinalEncoder()
        encoded_df[col + '_ord'] = oe.fit_transform(df[[col]].astype(str))
        encoding_report[col] = 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)'
encoding_report

{'order_date': 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)',
 'start_time': 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)',
 'typed_time': 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)',
 'at_plant_time': 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)',
 'name': 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)',
 'Nombre del proyecto': 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)',
 'ship_addr_line': 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)',
 'map_page': 'Ordinal encoding (high cardinality, consider target encoding for supervised tasks)'}

## 4. Numerical Scaling
Scaling is applied according to the planned model types. Here, StandardScaler is used for normally distributed features, RobustScaler for features with outliers.

In [8]:
# Example: Use RobustScaler for all numerical columns (since outliers were capped, but some may remain)
scaler = RobustScaler()
scaled_cols = [col for col in num_cols if col in encoded_df.columns]
encoded_df[scaled_cols] = scaler.fit_transform(encoded_df[scaled_cols])

## 5. Feature Engineering
Create new features if domain knowledge suggests useful combinations. Document each new feature.

In [9]:
# --- Feature Engineering for Volume per Hour Prediction ---
# Parse datetime columns
df['order_date'] = pd.to_datetime(df['order_date'])
df['start_time'] = pd.to_datetime(df['start_time'])
df['typed_time'] = pd.to_datetime(df['typed_time'])
df['at_plant_time'] = pd.to_datetime(df['at_plant_time'])

# Extract time features
df['hour'] = df['start_time'].dt.hour
df['dayofweek'] = df['start_time'].dt.dayofweek
df['month'] = df['start_time'].dt.month
df['day'] = df['start_time'].dt.day

# Calculate cycle time features
df['cycle_minutes'] = (df['at_plant_time'] - df['typed_time']).dt.total_seconds() / 60.0

# Sort for rolling features
df = df.sort_values(['order_date','start_time'])

# Rolling window features (per plant, per truck, per address)
df['vol_plant_1h'] = df.groupby('ship_plant_code')['u_Volumen'].rolling('1h', on='start_time').mean().reset_index(0,drop=True)
df['vol_truck_1h'] = df.groupby('truck_code')['u_Volumen'].rolling('1h', on='start_time').mean().reset_index(0,drop=True)
df['vol_addr_1h'] = df.groupby('ship_addr_line')['u_Volumen'].rolling('1h', on='start_time').mean().reset_index(0,drop=True)

# Lag features (previous delivery volume for truck/plant)
df['vol_truck_prev'] = df.groupby('truck_code')['u_Volumen'].shift(1)
df['vol_plant_prev'] = df.groupby('ship_plant_code')['u_Volumen'].shift(1)

# Aggregate features
truck_avg = df.groupby('truck_code')['u_Volumen'].transform('mean')
plant_avg = df.groupby('ship_plant_code')['u_Volumen'].transform('mean')
df['truck_avg_vol'] = truck_avg
df['plant_avg_vol'] = plant_avg

# Drop ID columns not useful for prediction
drop_cols = ['tkt_code','order_code','Nombre del proyecto','name','ship_addr_line']
df = df.drop(columns=drop_cols, errors='ignore')

# --- Categorical Encoding ---
# One-hot encode low-cardinality categoricals
df = pd.get_dummies(df, columns=['ship_plant_code','map_page','hour','dayofweek','month'], drop_first=True)
# Ordinal encode truck_code (high cardinality)
from sklearn.preprocessing import OrdinalEncoder
df['truck_code_ord'] = OrdinalEncoder().fit_transform(df[['truck_code']].astype(str))
df = df.drop(columns=['truck_code'], errors='ignore')

# --- Scaling ---
# StandardScaler for all numeric features (except target)
from sklearn.preprocessing import StandardScaler
feature_cols = [col for col in df.columns if col not in ['u_Volumen','order_date','start_time','typed_time','at_plant_time','day']]
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])



ValueError: invalid on specified as start_time, must be a column (of DataFrame), an Index or None

## 6. Train/Validation/Test Split (Optional)
Split the data for modeling, stratifying if classification. Save splits to `data/processed/`.

In [ ]:
# Example: 70/15/15 split, no stratification (replace with your target variable if classification)
train, temp = train_test_split(encoded_df, test_size=0.3, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)
print(f'Train: {train.shape}, Val: {val.shape}, Test: {test.shape}')
train.to_csv('../data/processed/train.csv', index=False)
val.to_csv('../data/processed/val.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

## 7. Save Final Clean Dataset
Save the fully processed dataset for modeling.

In [ ]:
encoded_df.to_csv('../data/processed/dataset_clean_v2.csv', index=False)